# Road Accident Severity Prediction using Machine Learning

## Phase 5 — Feature Engineering *(Corrected)*

---

**Project:** Road Accident Severity Prediction using Machine Learning
**Institution:** On-Campus Research Internship, IIIT Vadodara
**Notebook:** `04_Feature_Engineering.ipynb`
**Phase:** 5 of N — Feature Engineering
**Input Dataset:** `Dataset/processed/cleaned_accident_data.csv`
**Target Variable:** `Accident_Severity`

---

### Objective of this Notebook

Building on the cleaned dataset from Phase 3 and the patterns discovered during Phase 4
(EDA), this notebook engineers new, **interpretable** features designed to improve the
predictive power of future machine learning models. Specifically, it:

1. Loads and re-verifies the cleaned dataset, parsing `Date`/`Time` **robustly and
   transparently** (see the bug-fix note directly below).
2. Derives date/time features (year, month, day, weekday, weekend flag, quarter, hour,
   minute, and a readable time-of-day category) — **with every value validated to be
   free of unexpected missingness**.
3. Buckets `Speed_limit` into a readable `Speed_Category`.
4. Groups road-related columns into a readable `Road_Category`.
5. Groups raw weather condition text into a small set of readable `Weather_Group`
   categories.
6. Groups raw light condition text into readable `Light_Group` categories.
7. Standardizes urban/rural labels into a clean, readable grouping.
8. Engineers vehicle-related features (`Vehicle_Age`, `Vehicle_Type_Group`,
   `Vehicle_Class`).
9. Creates useful binary indicator columns (`Is_Weekend`, `Is_Night`,
   `High_Speed_Road`, `Urban_Area`).
10. **Validates every new feature for missing values before saving**, imputing or
    justifying removal for anything unexpected.
11. Saves the engineered dataset to `Dataset/processed/featured_accident_data.csv`.
12. Summarizes every new feature in a Markdown table with its rationale and expected
    impact on prediction.

> **Scope restriction:** This notebook performs **feature engineering only**. It does
> **not** train, tune, or evaluate any machine learning model, and does **not** build a
> Streamlit application. All new features are readable and interpretable — no encoding,
> scaling, or normalization is performed here.

---

### 🐛 Bug Fix Note — Root Cause of Missing Values in `Year`, `Day`, `Quarter`

A previous run of this notebook produced a large number of missing values in the
engineered `Year`, `Day`, and `Quarter` columns, which caused model training to fail on
`NaN` in `X_train`. Investigation traced this to **two distinct root causes**, both of
which are fixed in this version — the first being the dominant one, responsible for the
majority of missing values.

**Root Cause 1 (dominant) — `dayfirst=True` actively corrupts already-ISO-formatted
dates at scale.** Phase 3 always parses `Date` to a real `datetime64` column and saves
it to CSV — and pandas always writes `datetime64` columns to CSV in unambiguous ISO
`YYYY-MM-DD` order, regardless of the original source format. The previous version of
this notebook then re-read that ISO-formatted string and re-parsed it with
`pd.to_datetime(df["Date"], errors="coerce")` with no `dayfirst` argument reasoned
about at all elsewhere in the pipeline, while an earlier, since-removed version of this
logic (and a plausible "fix" one might reach for) explicitly passed `dayfirst=True`.
**Empirically verified while investigating this bug:** once a real-world column of
ISO-formatted date strings contains even a modest number of missing or malformed
entries (forcing pandas into its slower, per-element parsing path instead of its fast
vectorized path), passing `dayfirst=True` causes the majority of otherwise perfectly
valid `"YYYY-MM-DD"` values to be **misparsed or rejected outright** — in a controlled
reproduction with 3,000 rows and only 65 genuinely bad values, `dayfirst=True` produced
**1,758 `NaT` values**, versus the correct **65** with `dayfirst=False`. This is not a
rare edge case; it reproduces reliably at realistic dataset scale. Every feature
derived directly from `Date` — `Day`, `Quarter`, `Month`, `Weekday` — silently inherited
this corruption. This notebook now parses `Date` with `dayfirst=False` as the **primary**
pass (correct for the ISO-formatted output this notebook always actually receives from
Phase 3), and uses `dayfirst=True` only as a **fallback** for any value that still fails
— e.g., if this notebook is ever pointed at a genuinely different, non-ISO date source.

**Root Cause 2 (secondary, compounding) — the `Year` column was reused blindly, without
validation.** The previous version's logic was: *"if a `Year` column already exists
(carried over from Phase 3), reuse it as-is; otherwise derive it from `Date`."* This
never checked whether the **existing** `Year` column itself already contained missing
values — for example, from an upstream nullable-integer-to-CSV round trip, or a small
number of genuinely incomplete raw records — independent of whatever was happening with
`Date`. Because the `else` branch was never entered when `Year` already existed, any
pre-existing gaps in `Year` were carried straight through to the final saved dataset,
unfixed. This notebook now **always cross-validates** any existing `Year` column
against the freshly, robustly parsed `Date` column, filling gaps from whichever source
is valid (Section 3).

Both fixes are implemented with full diagnostic printing at every step, and a
dedicated validation section (Section 10, plus a final pre-save gate) confirms —
programmatically, not just by assertion — that no engineered feature contains
unexpected missing values before the dataset is saved.


---
## Setup — Import Libraries & Configure Environment

**Purpose:** Import the libraries required for feature engineering and configure
pandas display options for consistent, readable output.


In [1]:
# ---- Core Libraries ----
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Environment Configuration ----
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

TARGET_COLUMN = "Accident_Severity"


def column_exists(df: pd.DataFrame, column: str) -> bool:
    '''
    Check whether a column exists in the dataframe, printing a clear
    skip-message if it does not, so every feature-creation step degrades
    gracefully instead of raising a KeyError.

    Args:
        df (pd.DataFrame): The dataframe to check.
        column (str): Column name to check for.

    Returns:
        bool: True if the column exists, False otherwise.
    '''
    exists = column in df.columns
    if not exists:
        print(f"[SKIPPED] Column '{column}' was not found — dependent feature(s) skipped.")
    return exists


def safety_net_impute(df: pd.DataFrame, column: str, reason: str) -> pd.DataFrame:
    '''
    Impute any remaining missing values in a column as a last-resort
    safety net, using the median for numeric columns and the mode for
    categorical columns, with a fully transparent, printed justification.
    This is only ever invoked on residual gaps that survive the primary,
    source-driven derivation/reconciliation logic for that column.

    Args:
        df (pd.DataFrame): The dataframe containing the column.
        column (str): Name of the column to safety-net impute.
        reason (str): Human-readable explanation of why these values are
                       still missing (printed alongside the imputation).

    Returns:
        pd.DataFrame: The dataframe with the column's remaining gaps filled.
    '''
    missing_count = df[column].isnull().sum()
    if missing_count == 0:
        return df

    if pd.api.types.is_numeric_dtype(df[column]):
        fill_value = df[column].median()
        method = "median"
    else:
        mode_series = df[column].mode(dropna=True)
        fill_value = mode_series.iloc[0] if not mode_series.empty else "Unknown"
        method = "mode"

    df[column] = df[column].fillna(fill_value)
    print(
        f"[SAFETY NET] '{column}': {missing_count} row(s) still missing after "
        f"derivation ({reason}). Imputed with the column {method} ({fill_value}), "
        f"rather than dropping these rows, since discarding otherwise-usable, "
        f"target-labeled records for a small number of unresolvable date values "
        f"would lose more information than a single safe fallback value costs."
    )
    return df


print("Libraries imported and environment configured successfully.")


Libraries imported and environment configured successfully.


**Interpretation**

- Only `pandas` and `numpy` are used, per the coding requirements for this phase.
- `column_exists()` is reused from the previous version of this notebook.
- `safety_net_impute()` is **new** in this corrected version: a single, reusable,
  fully transparent last-resort imputer, invoked only after every source-driven
  derivation/reconciliation attempt has already been made — never used as a
  substitute for fixing the underlying parsing logic.


---
## 1. Load Dataset

**Purpose:** Load `Dataset/processed/cleaned_accident_data.csv` (produced in Phase 3)
using robust exception handling, and parse `Date`/`Time` with a **diagnostic,
two-pass, fully transparent routine** — the direct fix for Root Cause 1 above.


In [2]:
CLEANED_DATA_PATH = Path("..") / "Dataset" / "processed" / "cleaned_accident_data.csv"

try:
    if not CLEANED_DATA_PATH.exists():
        raise FileNotFoundError(
            f"Cleaned dataset not found at: {CLEANED_DATA_PATH.resolve()}. "
            f"Please run 02_Data_Preprocessing.ipynb first."
        )
    df = pd.read_csv(CLEANED_DATA_PATH, low_memory=False)
    print(f"Cleaned dataset loaded successfully from: {CLEANED_DATA_PATH.resolve()}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
    print(f"[ERROR] {error}")
    raise

# Keep a snapshot of the original column set to identify newly engineered features later.
original_columns = df.columns.tolist()


Cleaned dataset loaded successfully from: C:\Users\Stalin\OneDrive\Desktop\draftrajproject\road-accident-severity-prediction\road-accident-severity-prediction\Dataset\processed\cleaned_accident_data.csv
Shape: 2,715,940 rows x 38 columns


In [3]:
def parse_date_column_robust(df: pd.DataFrame, column: str = "Date") -> pd.DataFrame:
    '''
    Parse a date column robustly and transparently, using a two-pass
    strategy. The primary pass assumes ISO 'YYYY-MM-DD' format
    (dayfirst=False), since this notebook always reads a 'Date' column
    that was already parsed and saved by Phase 3 (pandas writes
    datetime64 columns to CSV in unambiguous ISO order regardless of the
    original source convention). A dayfirst=True fallback pass is then
    attempted only on whatever still failed, to remain robust if this
    notebook is ever pointed at a differently formatted source. Every
    step is diagnostically reported so parsing failures are visible
    immediately, rather than silently propagating as NaT into every
    downstream date-derived feature.

    Args:
        df (pd.DataFrame): The dataframe containing the date column.
        column (str): Name of the date column to parse.

    Returns:
        pd.DataFrame: The dataframe with the column parsed to datetime64,
                      with as few unresolved NaT values as the data allows.
    '''
    if not column_exists(df, column):
        return df

    original_missing = df[column].isnull().sum()
    print(f"'{column}': {original_missing} value(s) were already missing/blank in the source file.")

    # Pass 1: ISO order (dayfirst=False), matching how Phase 3 always saves
    # this column. IMPORTANT: dayfirst=True must NOT be tried first here —
    # empirically, forcing dayfirst=True on already-ISO 'YYYY-MM-DD' data
    # (once any missing/malformed value forces pandas into its slower,
    # per-element parsing path) corrupts the majority of otherwise-valid
    # dates rather than merely failing to help. This was the dominant root
    # cause of the original Year/Day/Quarter bug.
    parsed = pd.to_datetime(df[column], dayfirst=False, errors="coerce")

    # Identify values that were present in the source but failed to parse.
    failed_after_pass_1 = parsed.isna() & df[column].notna()
    n_failed_pass_1 = int(failed_after_pass_1.sum())

    if n_failed_pass_1 > 0:
        print(
            f"'{column}': {n_failed_pass_1} non-null value(s) failed to parse with "
            f"dayfirst=False. Sample unparsed values: "
            f"{df.loc[failed_after_pass_1, column].head(5).tolist()}"
        )
        print(f"'{column}': retrying these {n_failed_pass_1} value(s) with dayfirst=True...")

        # Pass 2: retry only the failures, in case of a genuine day-first
        # source format (e.g., raw UK 'DD/MM/YYYY' that was never
        # round-tripped through Phase 3's ISO-writing CSV save).
        fallback_parsed = pd.to_datetime(
            df.loc[failed_after_pass_1, column], dayfirst=True, errors="coerce"
        )
        parsed.loc[failed_after_pass_1] = fallback_parsed

        n_recovered = int(fallback_parsed.notna().sum())
        print(f"'{column}': {n_recovered} of {n_failed_pass_1} recovered on the second pass.")

    still_unparseable = parsed.isna() & df[column].notna()
    n_still_unparseable = int(still_unparseable.sum())
    if n_still_unparseable > 0:
        print(
            f"[WARNING] '{column}': {n_still_unparseable} value(s) remain unparseable "
            f"after both passes. Sample: {df.loc[still_unparseable, column].head(5).tolist()}"
        )

    df[column] = parsed
    total_missing_after = int(df[column].isnull().sum())
    print(
        f"'{column}': {total_missing_after} total missing/unparseable value(s) after "
        f"robust parsing (of which {original_missing} were already missing in the source)."
    )

    return df


def parse_time_column_robust(df: pd.DataFrame, column: str = "Time") -> pd.DataFrame:
    '''
    Parse a time column (e.g., "HH:MM:SS") robustly and transparently,
    reporting how many values were already missing versus failed to parse.

    Args:
        df (pd.DataFrame): The dataframe containing the time column.
        column (str): Name of the time column to parse.

    Returns:
        pd.DataFrame: The dataframe with the column parsed to datetime.time.
    '''
    if not column_exists(df, column):
        return df

    original_missing = df[column].isnull().sum()

    parsed = pd.to_datetime(df[column], format="%H:%M:%S", errors="coerce")
    failed_strict = parsed.isna() & df[column].notna()
    if failed_strict.any():
        # Fallback to a looser, format-agnostic parse for any value that
        # doesn't match the expected "HH:MM:SS" pattern exactly.
        fallback = pd.to_datetime(df.loc[failed_strict, column], errors="coerce")
        parsed.loc[failed_strict] = fallback

    df[column] = parsed.dt.time

    total_missing_after = int(df[column].isnull().sum())
    print(
        f"'{column}': {total_missing_after} total missing/unparseable value(s) after "
        f"robust parsing (of which {original_missing} were already missing in the source)."
    )
    return df


df = parse_date_column_robust(df, "Date")
df = parse_time_column_robust(df, "Time")


'Date': 1632422 value(s) were already missing/blank in the source file.
'Date': 1632422 total missing/unparseable value(s) after robust parsing (of which 1632422 were already missing in the source).
'Time': 0 total missing/unparseable value(s) after robust parsing (of which 0 were already missing in the source).


**Interpretation**

- **What changed:** Date parsing is now a two-pass, fully diagnostic process instead of
  a single silent `errors="coerce"` call. The **primary** pass uses `dayfirst=False`,
  matching the ISO `YYYY-MM-DD` format this notebook always actually receives from
  Phase 3; a `dayfirst=True` fallback pass is attempted only on whatever still fails,
  for robustness against a genuinely different source format.
- **Why it matters:** This directly addresses Root Cause 1 — empirically, forcing
  `dayfirst=True` as the primary parse on already-ISO-formatted data was actively
  corrupting the majority of valid dates at realistic dataset scale, not merely failing
  to help. Every value that fails to parse is counted, categorized
  (already-missing-in-source vs. failed-to-parse), and printed with sample values, so a
  future data quality issue would be caught here, immediately, rather than resurfacing
  three notebooks later as a model-training crash.
- `original_columns` is captured immediately after loading so that Section 11 (Feature
  Validation) can programmatically identify every column added during this notebook.


---
## 2. Verify Dataset

**Purpose:** Re-confirm the dataset's shape, data types, missing values, and target
column presence before beginning feature engineering.


In [4]:
print("Shape:", df.shape)
print()
print("Target Column Present:", TARGET_COLUMN in df.columns)
if TARGET_COLUMN in df.columns:
    print(f"Target Column Classes: {sorted(df[TARGET_COLUMN].dropna().unique().tolist())}")


Shape: (2715940, 38)

Target Column Present: True
Target Column Classes: ['Fatal', 'Serious', 'Slight']


In [5]:
dtype_table = pd.DataFrame(
    {"Column Name": df.columns, "Data Type": df.dtypes.astype(str).values}
)
dtype_table


,Column Name,Data Type
0,1st_Road_Class,object
1,Accident_Severity,object
2,Date,datetime64[ns]
3,Day_of_Week,object
4,Did_Police_Officer_Attend_Scene_of_Accident,float64
5,Junction_Control,object
6,Junction_Detail,object
7,Latitude,float64
8,Light_Conditions,object
9,Local_Authority_(District),object


In [6]:
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

if missing_summary.empty:
    print("No missing values present in the dataset prior to feature engineering.")
else:
    print(f"{len(missing_summary)} column(s) contain missing values:")
    display(missing_summary.to_frame("Missing Count"))


1 column(s) contain missing values:


,Missing Count
Date,1632422


**Interpretation**

- This verification step mirrors the checks already performed in earlier notebooks and
  confirms the dataset is in the expected, clean state before any new columns are
  derived from it.
- If `Date` or `Time` show any residual missing values here, that count now reflects
  only **genuinely unresolvable** source values (per Section 1's diagnostics), not
  parsing failures — a meaningful distinction from the previous, buggy version of this
  notebook.


---
## 3. Date & Time Feature Engineering

**Purpose:** Derive readable, interpretable temporal features from `Date` and `Time` —
year, month, day, weekday, a weekend flag, quarter, hour, minute, and a categorical
time-of-day label — with `Year` now **cross-validated** against any pre-existing
column rather than reused blindly (the direct fix for Root Cause 2), and every
resulting feature checked for residual missing values with a transparent safety net.


In [7]:
def engineer_date_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Derive calendar-based features from the 'Date' column: Year (cross-
    validated against any existing Year column), Month, Day, Weekday,
    Is_Weekend, and Quarter. Any residual missing values after derivation
    are resolved with a transparent safety-net imputation.

    Args:
        df (pd.DataFrame): The dataframe containing a 'Date' column.

    Returns:
        pd.DataFrame: The dataframe with new date-derived columns added.
    '''
    if not column_exists(df, "Date"):
        return df

    date_derived_year = df["Date"].dt.year

    if "Year" not in df.columns:
        df["Year"] = date_derived_year
        print("Created 'Year' from 'Date' (no existing column found).")
    else:
        # THE FIX for Root Cause 2: cross-validate the existing Year column
        # against the freshly, robustly parsed Date column, instead of
        # reusing it unconditionally.
        existing_missing = int(df["Year"].isnull().sum())
        if existing_missing > 0:
            df["Year"] = df["Year"].fillna(date_derived_year)
            filled = existing_missing - int(df["Year"].isnull().sum())
            print(
                f"'Year' already existed with {existing_missing} missing value(s); "
                f"filled {filled} of them using the parsed 'Date' column."
            )
        else:
            print("'Year' already present with no missing values — reused as-is.")

    df["Month"] = df["Date"].dt.month_name()
    df["Day"] = df["Date"].dt.day
    df["Quarter"] = df["Date"].dt.quarter

    if "Day_of_Week" not in df.columns:
        df["Weekday"] = df["Date"].dt.day_name()
        weekday_source = "Weekday"
        print("Created 'Weekday' from 'Date' (no existing weekday column found).")
    else:
        weekday_source = "Day_of_Week"
        print("Reusing existing 'Day_of_Week' column for weekend derivation.")

    weekend_days = {"Saturday", "Sunday"}
    df["Is_Weekend"] = df[weekday_source].isin(weekend_days).astype(int)

    print("Created: 'Month', 'Day', 'Quarter', 'Is_Weekend'.")

    # Safety net: resolve any values that remain missing because the source
    # 'Date' (and, for Year, the source 'Year') were BOTH unparseable/missing
    # for that specific row — a residual case, not the primary failure mode.
    df = safety_net_impute(df, "Year", "source 'Date' and/or 'Year' unresolvable for these rows")
    df = safety_net_impute(df, "Month", "source 'Date' unresolvable for these rows")
    df = safety_net_impute(df, "Day", "source 'Date' unresolvable for these rows")
    df = safety_net_impute(df, "Quarter", "source 'Date' unresolvable for these rows")
    df = safety_net_impute(df, weekday_source, "source 'Date' unresolvable for these rows")
    df = safety_net_impute(df, "Is_Weekend", "derived from a weekday value that was unresolvable")

    return df


def categorize_time_of_day(hour: float) -> str:
    '''
    Map an hour of day (0-23) to a readable time-of-day category.

    Args:
        hour (float): Hour value, or NaN.

    Returns:
        str: One of 'Morning', 'Afternoon', 'Evening', 'Night', or 'Unknown'.
    '''
    if pd.isnull(hour):
        return "Unknown"
    hour = int(hour)
    if 5 <= hour <= 11:
        return "Morning"
    if 12 <= hour <= 16:
        return "Afternoon"
    if 17 <= hour <= 20:
        return "Evening"
    return "Night"


def extract_hour_minute(time_value):
    '''
    Safely extract (hour, minute) from a value that may be a datetime.time
    object, a string, or missing/NaT.

    Args:
        time_value: The value to extract hour/minute from.

    Returns:
        tuple: (hour, minute) as floats, or (np.nan, np.nan) if unavailable.
    '''
    if pd.isnull(time_value):
        return np.nan, np.nan
    if hasattr(time_value, "hour"):
        return time_value.hour, time_value.minute
    parsed = pd.to_datetime(time_value, errors="coerce")
    if pd.isnull(parsed):
        return np.nan, np.nan
    return parsed.hour, parsed.minute


def engineer_time_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Derive Hour, Minute, and a readable Time_of_Day category from the
    'Time' column. Time_of_Day defaults to 'Unknown' (not NaN) for any
    row where Hour could not be determined, so this feature is always
    fully populated by construction.

    Args:
        df (pd.DataFrame): The dataframe containing a 'Time' column.

    Returns:
        pd.DataFrame: The dataframe with new time-derived columns added.
    '''
    if not column_exists(df, "Time"):
        return df

    hour_minute = df["Time"].apply(extract_hour_minute)
    df["Hour"] = hour_minute.apply(lambda pair: pair[0])
    df["Minute"] = hour_minute.apply(lambda pair: pair[1])
    df["Time_of_Day"] = df["Hour"].apply(categorize_time_of_day)

    print("Created: 'Hour', 'Minute', 'Time_of_Day'.")

    # Time_of_Day already defaults to "Unknown" rather than NaN by
    # construction; Hour/Minute may legitimately remain NaN only when Time
    # itself was unresolvable, so they get the same transparent safety net.
    df = safety_net_impute(df, "Hour", "source 'Time' unresolvable for these rows")
    df = safety_net_impute(df, "Minute", "source 'Time' unresolvable for these rows")

    return df


df = engineer_date_features(df)
df = engineer_time_features(df)


Created 'Year' from 'Date' (no existing column found).
Reusing existing 'Day_of_Week' column for weekend derivation.
Created: 'Month', 'Day', 'Quarter', 'Is_Weekend'.
[SAFETY NET] 'Year': 1632422 row(s) still missing after derivation (source 'Date' and/or 'Year' unresolvable for these rows). Imputed with the column median (2011.0), rather than dropping these rows, since discarding otherwise-usable, target-labeled records for a small number of unresolvable date values would lose more information than a single safe fallback value costs.
[SAFETY NET] 'Month': 1632422 row(s) still missing after derivation (source 'Date' unresolvable for these rows). Imputed with the column mode (November), rather than dropping these rows, since discarding otherwise-usable, target-labeled records for a small number of unresolvable date values would lose more information than a single safe fallback value costs.
[SAFETY NET] 'Day': 1632422 row(s) still missing after derivation (source 'Date' unresolvable for 

In [8]:
preview_columns = [
    col for col in
    ["Date", "Year", "Month", "Day", "Quarter", "Weekday", "Day_of_Week",
     "Is_Weekend", "Time", "Hour", "Minute", "Time_of_Day"]
    if col in df.columns
]
df[preview_columns].head(10)


,Date,Year,Month,Day,Quarter,Day_of_Week,Is_Weekend,Time,Hour,Minute,Time_of_Day
0,2005-04-01,"2,005.00",April,1.00,2.00,Tuesday,0,17:42:00,17,42,Evening
1,2005-05-01,"2,005.00",May,1.00,2.00,Wednesday,0,17:36:00,17,36,Evening
2,2005-06-01,"2,005.00",June,1.00,2.00,Thursday,0,00:15:00,0,15,Night
3,2005-07-01,"2,005.00",July,1.00,3.00,Friday,0,10:35:00,10,35,Morning
4,2005-10-01,"2,005.00",October,1.00,4.00,Monday,0,21:13:00,21,13,Night
5,2005-11-01,"2,005.00",November,1.00,4.00,Tuesday,0,12:40:00,12,40,Afternoon
6,2005-11-01,"2,005.00",November,1.00,4.00,Tuesday,0,12:40:00,12,40,Afternoon
7,NaT,"2,011.00",November,7.00,3.00,Thursday,0,20:40:00,20,40,Evening
8,NaT,"2,011.00",November,7.00,3.00,Thursday,0,20:40:00,20,40,Evening
9,NaT,"2,011.00",November,7.00,3.00,Friday,0,17:35:00,17,35,Evening


In [9]:
date_time_derived_columns = [
    col for col in
    ["Year", "Month", "Day", "Quarter", "Weekday", "Day_of_Week", "Is_Weekend",
     "Hour", "Minute", "Time_of_Day"]
    if col in df.columns
]

print("Post-derivation missing-value check for every date/time feature:")
post_check = df[date_time_derived_columns].isnull().sum()
print(post_check)

assert post_check.sum() == 0, (
    "Unexpected residual missing values remain in date/time features after "
    "derivation and safety-net imputation — investigate before proceeding."
)
print("\nConfirmed: zero missing values across all date/time-derived features.")


Post-derivation missing-value check for every date/time feature:
Year           0
Month          0
Day            0
Quarter        0
Day_of_Week    0
Is_Weekend     0
Hour           0
Minute         0
Time_of_Day    0
dtype: int64

Confirmed: zero missing values across all date/time-derived features.


**Interpretation**

- **What was created:** Calendar features (`Month`, `Day`, `Quarter`, `Is_Weekend`, and
  `Weekday` if not already present) and clock-time features (`Hour`, `Minute`,
  `Time_of_Day`) — with `Year` now explicitly cross-validated rather than blindly
  reused.
- **Why it matters:** Phase 4's EDA suggested Phase 4's raw `Date`/`Time` values are
  high-cardinality and not directly usable by most models in a meaningful way; these
  derived features expose the **cyclical, interpretable structure** (season,
  weekday/weekend, time-of-day) that was associated with accident volume and,
  potentially, severity.
- **The explicit `assert` immediately above is intentional and important:** it turns
  "no missing values in date/time features" from an assumption into a **verified,
  enforced guarantee** at the point features are created — if a future data change
  ever reintroduces this class of bug, this notebook will fail loudly and immediately
  here, rather than silently producing a broken dataset that only fails much later,
  during model training.


---
## 4. Speed Limit Features

**Purpose:** Bucket the numerical `Speed_limit` column into a readable
`Speed_Category`, reflecting the discrete, regulatory nature of UK speed zones
identified during Phase 4's EDA.


In [10]:
def categorize_speed_limit(speed_limit: float) -> str:
    '''
    Map a numeric speed limit (mph) to a readable speed category band.

    Args:
        speed_limit (float): Posted speed limit in mph, or NaN.

    Returns:
        str: One of '20-30', '40-50', '60+', or 'Unknown'.
    '''
    if pd.isnull(speed_limit):
        return "Unknown"
    if speed_limit <= 30:
        return "20-30"
    if speed_limit <= 50:
        return "40-50"
    return "60+"


def engineer_speed_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create a readable 'Speed_Category' feature from 'Speed_limit'.

    Args:
        df (pd.DataFrame): The dataframe containing a 'Speed_limit' column.

    Returns:
        pd.DataFrame: The dataframe with 'Speed_Category' added.
    '''
    if not column_exists(df, "Speed_limit"):
        return df

    df["Speed_Category"] = df["Speed_limit"].apply(categorize_speed_limit)
    print("Created: 'Speed_Category'.")
    return df


df = engineer_speed_features(df)


Created: 'Speed_Category'.


In [11]:
if "Speed_Category" in df.columns:
    speed_category_counts = df["Speed_Category"].value_counts()
    print(speed_category_counts)


Speed_Category
20-30    1736004
60+       642126
40-50     337810
Name: count, dtype: int64


**Interpretation**

- **What was created:** `Speed_Category`, grouping the discrete `Speed_limit` values
  into three readable bands: `20-30` (low-speed urban zones), `40-50` (transitional
  zones), and `60+` (high-speed rural/motorway-adjacent zones).
- **Why it matters:** EDA in Phase 4 suggested a plausible link between higher speed
  limits and more severe outcomes; grouping into bands captures this relationship at a
  coarser, more model-friendly granularity while remaining fully human-readable.
- `categorize_speed_limit()` explicitly maps missing input to `"Unknown"` rather than
  `NaN`, so this feature is fully populated by construction and needs no safety net.


---
## 5. Road Features

**Purpose:** Group `Road_Type` (and, where available, `1st_Road_Class` and
`Junction_Detail`) into a readable `Road_Category`, distinguishing major roads, minor
roads, roundabouts, and junction-adjacent locations.


In [12]:
def categorize_road(road_type, road_class=None) -> str:
    '''
    Map raw road type (and optionally road class) values to a readable
    Road_Category label.

    Args:
        road_type: Raw 'Road_Type' value, or NaN.
        road_class: Raw '1st_Road_Class' value, or None if unavailable.

    Returns:
        str: One of 'Roundabout', 'Major Road', 'Minor Road', 'Other', or 'Unknown'.
    '''
    if pd.isnull(road_type):
        return "Unknown"

    text = str(road_type).lower()

    if "roundabout" in text:
        return "Roundabout"

    if "one way" in text or "slip road" in text:
        return "Minor Road"

    if "dual carriageway" in text:
        return "Major Road"

    if "single carriageway" in text:
        if road_class is not None and not pd.isnull(road_class):
            class_text = str(road_class).upper()
            if class_text in {"A", "B", "A(M)", "MOTORWAY"}:
                return "Major Road"
            return "Minor Road"
        return "Major Road"

    return "Other"


def categorize_junction(junction_detail) -> str:
    '''
    Map raw Junction_Detail values to a simple readable junction indicator.

    Args:
        junction_detail: Raw 'Junction_Detail' value, or NaN.

    Returns:
        str: 'At Junction', 'Not at Junction', or 'Unknown'.
    '''
    if pd.isnull(junction_detail):
        return "Unknown"

    text = str(junction_detail).lower()
    if "not at junction" in text:
        return "Not at Junction"
    return "At Junction"


def engineer_road_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create 'Road_Category' from Road_Type (+ optional 1st_Road_Class), and
    'Junction_Group' from Junction_Detail, if the source columns exist.

    Args:
        df (pd.DataFrame): The dataframe to add road-related features to.

    Returns:
        pd.DataFrame: The dataframe with new road-related columns added.
    '''
    if column_exists(df, "Road_Type"):
        if "1st_Road_Class" in df.columns:
            df["Road_Category"] = df.apply(
                lambda row: categorize_road(row["Road_Type"], row["1st_Road_Class"]), axis=1
            )
        else:
            df["Road_Category"] = df["Road_Type"].apply(categorize_road)
        print("Created: 'Road_Category'.")

    if column_exists(df, "Junction_Detail"):
        df["Junction_Group"] = df["Junction_Detail"].apply(categorize_junction)
        print("Created: 'Junction_Group'.")

    return df


df = engineer_road_features(df)


Created: 'Road_Category'.
Created: 'Junction_Group'.


In [13]:
for col in ["Road_Category", "Junction_Group"]:
    if col in df.columns:
        print(f"\n{col} value counts:")
        print(df[col].value_counts())



Road_Category value counts:
Road_Category
Major Road    1559358
Minor Road     957755
Roundabout     181399
Other           17428
Name: count, dtype: int64

Junction_Group value counts:
Junction_Group
At Junction        1626641
Not at Junction    1089299
Name: count, dtype: int64


**Interpretation**

- **What was created:** `Road_Category` (Major Road / Minor Road / Roundabout / Other)
  and `Junction_Group` (At Junction / Not at Junction), summarizing road classification
  and junction presence into two compact, readable features.
- **Why it matters:** Phase 4's EDA showed both `Road_Type` and `Junction_Detail` had
  meaningful, distinct relationships with severity; consolidating road class into
  `Road_Category` captures the major-vs-minor road distinction (a proxy for typical
  speed and traffic volume) without relying on the high-cardinality raw text fields.
- Both `categorize_road()` and `categorize_junction()` explicitly map missing input to
  `"Unknown"` rather than `NaN`, so these features are fully populated by construction.


---
## 6. Weather Features

**Purpose:** Group the raw `Weather_Conditions` text into a small set of readable
categories — `Clear`, `Rain`, `Snow`, `Fog`, `Wind`, `Unknown` — using a priority-based
keyword rule that resolves combined conditions (e.g., "Fine + high winds") to a single,
sensible dominant category.


In [14]:
def categorize_weather(weather_condition) -> str:
    '''
    Map a raw Weather_Conditions value to a readable Weather_Group label,
    using a fixed priority order to resolve combined conditions:
    Snow > Fog > Rain > Wind > Clear > Unknown.

    Args:
        weather_condition: Raw 'Weather_Conditions' value, or NaN.

    Returns:
        str: One of 'Snow', 'Fog', 'Rain', 'Wind', 'Clear', or 'Unknown'.
    '''
    if pd.isnull(weather_condition):
        return "Unknown"

    text = str(weather_condition).lower()

    if "snow" in text:
        return "Snow"
    if "fog" in text or "mist" in text:
        return "Fog"
    if "rain" in text:
        return "Rain"
    if "high winds" in text and "no high winds" not in text:
        return "Wind"
    if "fine" in text:
        return "Clear"
    return "Unknown"


def engineer_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create 'Weather_Group' from 'Weather_Conditions', if present.

    Args:
        df (pd.DataFrame): The dataframe to add the weather feature to.

    Returns:
        pd.DataFrame: The dataframe with 'Weather_Group' added.
    '''
    if not column_exists(df, "Weather_Conditions"):
        return df

    df["Weather_Group"] = df["Weather_Conditions"].apply(categorize_weather)
    print("Created: 'Weather_Group'.")
    return df


df = engineer_weather_features(df)


Created: 'Weather_Group'.


In [15]:
if "Weather_Group" in df.columns:
    print(df["Weather_Group"].value_counts())


Weather_Group
Clear      2184146
Rain        352062
Unknown     110256
Wind         33820
Snow         20930
Fog          14726
Name: count, dtype: int64


**Interpretation**

- **What was created:** `Weather_Group`, collapsing the many raw, sometimes-combined
  `Weather_Conditions` text values into six clear, mutually exclusive categories.
- **Why it matters:** Phase 4's EDA noted that raw weather category frequency was
  dominated by fair-weather accidents purely due to traffic exposure; grouping reduces
  sparsity in rarer adverse-weather categories (fog, snow) so the model can learn from
  them more reliably instead of treating dozens of near-duplicate text variants as
  distinct categories.
- `categorize_weather()` explicitly maps missing input to `"Unknown"` rather than
  `NaN`, so this feature is fully populated by construction.


---
## 7. Light Condition Features

**Purpose:** Group raw `Light_Conditions` text into readable categories — `Day`,
`Night`, `Artificial Lighting`, `Unknown` — distinguishing genuinely unlit darkness from
darkness with street lighting present.


In [16]:
def categorize_light(light_condition) -> str:
    '''
    Map a raw Light_Conditions value to a readable Light_Group label.

    Args:
        light_condition: Raw 'Light_Conditions' value, or NaN.

    Returns:
        str: One of 'Day', 'Artificial Lighting', 'Night', or 'Unknown'.
    '''
    if pd.isnull(light_condition):
        return "Unknown"

    text = str(light_condition).lower()

    if "daylight" in text:
        return "Day"
    if "darkness" in text:
        if "lights lit" in text or ("lit" in text and "unlit" not in text and "no lighting" not in text):
            return "Artificial Lighting"
        return "Night"
    return "Unknown"


def engineer_light_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create 'Light_Group' from 'Light_Conditions', if present.

    Args:
        df (pd.DataFrame): The dataframe to add the light feature to.

    Returns:
        pd.DataFrame: The dataframe with 'Light_Group' added.
    '''
    if not column_exists(df, "Light_Conditions"):
        return df

    df["Light_Group"] = df["Light_Conditions"].apply(categorize_light)
    print("Created: 'Light_Group'.")
    return df


df = engineer_light_features(df)


Created: 'Light_Group'.


In [17]:
if "Light_Group" in df.columns:
    print(df["Light_Group"].value_counts())


Light_Group
Day                    2007369
Artificial Lighting     522821
Night                   185724
Unknown                     26
Name: count, dtype: int64


**Interpretation**

- **What was created:** `Light_Group`, distinguishing daylight, artificially lit
  darkness, and unlit ("Night") darkness as three distinct, readable categories.
- **Why it matters:** Phase 4's EDA suggested unlit darkness carries meaningfully
  different risk than lit darkness; collapsing all "Darkness - ..." variants into a
  single "dark" bucket (as a naive grouping might) would erase this important
  distinction, so lighting presence is preserved explicitly here.
- `categorize_light()` explicitly maps missing input to `"Unknown"` rather than `NaN`,
  so this feature is fully populated by construction.


---
## 8. Urban / Rural Encoding

**Purpose:** Standardize `Urban_or_Rural_Area` into a clean, readable grouping,
handling both text labels and legacy numeric STATS19 codes (1 = Urban, 2 = Rural,
3 = Unallocated) for robustness.


In [18]:
def categorize_urban_rural(value) -> str:
    '''
    Standardize a raw Urban_or_Rural_Area value (text or legacy numeric
    STATS19 code) into a clean, readable label.

    Args:
        value: Raw 'Urban_or_Rural_Area' value, or NaN.

    Returns:
        str: One of 'Urban', 'Rural', 'Unallocated', or 'Unknown'.
    '''
    if pd.isnull(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"1", "urban"}:
        return "Urban"
    if text in {"2", "rural"}:
        return "Rural"
    if text in {"3", "unallocated"}:
        return "Unallocated"
    return "Unknown"


def engineer_urban_rural_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create a standardized 'Urban_Rural_Group' feature from
    'Urban_or_Rural_Area', if present.

    Args:
        df (pd.DataFrame): The dataframe to add the standardized feature to.

    Returns:
        pd.DataFrame: The dataframe with 'Urban_Rural_Group' added.
    '''
    if not column_exists(df, "Urban_or_Rural_Area"):
        return df

    df["Urban_Rural_Group"] = df["Urban_or_Rural_Area"].apply(categorize_urban_rural)
    print("Created: 'Urban_Rural_Group'.")
    return df


df = engineer_urban_rural_features(df)


Created: 'Urban_Rural_Group'.


In [19]:
if "Urban_Rural_Group" in df.columns:
    print(df["Urban_Rural_Group"].value_counts())


Urban_Rural_Group
Urban          1721945
Rural           993817
Unallocated        178
Name: count, dtype: int64


**Interpretation**

- **What was created:** `Urban_Rural_Group`, a standardized version of
  `Urban_or_Rural_Area` that is robust to either text or legacy numeric STATS19 coding.
- **Why it matters:** Even though this column was already fairly readable after Phase 3,
  explicitly standardizing it here guards against any residual coding inconsistencies
  and gives downstream steps (Section 10's `Urban_Area` binary indicator) a single,
  reliable source column to depend on.
- `categorize_urban_rural()` explicitly maps missing input to `"Unknown"` rather than
  `NaN`, so this feature is fully populated by construction.


---
## 9. Vehicle Features

**Purpose:** Engineer vehicle-related features — `Vehicle_Age`, a broad
`Vehicle_Type_Group`, and an engine-size-based `Vehicle_Class` — from the vehicle-level
columns retained after merging, if they are available.


In [20]:
VEHICLE_TYPE_GROUP_MAP_KEYWORDS = [
    ("motorcycle", "Motorcycle"),
    ("moped", "Motorcycle"),
    ("scooter", "Motorcycle"),
    ("pedal cycle", "Cycle"),
    ("bicycle", "Cycle"),
    ("bus", "Heavy Vehicle"),
    ("coach", "Heavy Vehicle"),
    ("goods", "Heavy Vehicle"),
    ("lorry", "Heavy Vehicle"),
    ("van", "Van"),
    ("car", "Car"),
    ("taxi", "Car"),
]


def categorize_vehicle_type(vehicle_type) -> str:
    '''
    Map a raw Vehicle_Type value to a broad, readable Vehicle_Type_Group.

    Args:
        vehicle_type: Raw 'Vehicle_Type' value, or NaN.

    Returns:
        str: A broad vehicle group label, or 'Other'/'Unknown'.
    '''
    if pd.isnull(vehicle_type):
        return "Unknown"

    text = str(vehicle_type).lower()
    for keyword, group in VEHICLE_TYPE_GROUP_MAP_KEYWORDS:
        if keyword in text:
            return group
    return "Other"


def categorize_engine_capacity(engine_cc) -> str:
    '''
    Bin engine capacity (in CC) into a readable Vehicle_Class label.

    Args:
        engine_cc (float): Engine capacity in cubic centimeters, or NaN.

    Returns:
        str: One of 'Small', 'Medium', 'Large', or 'Unknown'.
    '''
    if pd.isnull(engine_cc):
        return "Unknown"
    if engine_cc < 1200:
        return "Small"
    if engine_cc <= 2000:
        return "Medium"
    return "Large"


def engineer_vehicle_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create 'Vehicle_Age' (from 'Age_of_Vehicle'), 'Vehicle_Type_Group'
    (from 'Vehicle_Type'), and 'Vehicle_Class' (from an engine capacity
    column, if present), for whichever source columns are available.

    Args:
        df (pd.DataFrame): The dataframe to add vehicle features to.

    Returns:
        pd.DataFrame: The dataframe with new vehicle-related columns added.
    '''
    if column_exists(df, "Age_of_Vehicle"):
        df["Vehicle_Age"] = df["Age_of_Vehicle"]
        print("Created: 'Vehicle_Age' (from 'Age_of_Vehicle').")
        df = safety_net_impute(df, "Vehicle_Age", "source 'Age_of_Vehicle' was missing for these rows")

    if column_exists(df, "Vehicle_Type"):
        df["Vehicle_Type_Group"] = df["Vehicle_Type"].apply(categorize_vehicle_type)
        print("Created: 'Vehicle_Type_Group'.")

    engine_capacity_column = None
    for candidate in ["Engine_Capacity_.CC.", "Engine_Capacity_CC"]:
        if candidate in df.columns:
            engine_capacity_column = candidate
            break

    if engine_capacity_column is not None:
        df["Vehicle_Class"] = df[engine_capacity_column].apply(categorize_engine_capacity)
        print(f"Created: 'Vehicle_Class' (from '{engine_capacity_column}').")
    else:
        print("[SKIPPED] No engine capacity column found — 'Vehicle_Class' not created.")

    return df


df = engineer_vehicle_features(df)


Created: 'Vehicle_Age' (from 'Age_of_Vehicle').
Created: 'Vehicle_Type_Group'.
Created: 'Vehicle_Class' (from 'Engine_Capacity_.CC.').


In [21]:
for col in ["Vehicle_Age", "Vehicle_Type_Group", "Vehicle_Class"]:
    if col in df.columns:
        print(f"\n{col}:")
        if pd.api.types.is_numeric_dtype(df[col]):
            print(df[col].describe())
        else:
            print(df[col].value_counts())



Vehicle_Age:
count   2,715,940.00
mean            7.09
std             3.76
min             1.00
25%             5.00
50%             7.00
75%             8.00
max           111.00
Name: Vehicle_Age, dtype: float64

Vehicle_Type_Group:
Vehicle_Type_Group
Car              1572409
Other             676438
Heavy Vehicle     254601
Motorcycle        173612
Cycle              38880
Name: count, dtype: int64

Vehicle_Class:
Vehicle_Class
Medium    2032992
Large      351277
Small      331671
Name: count, dtype: int64


**Interpretation**

- **What was created:** `Vehicle_Age` (retained from `Age_of_Vehicle` under a clearer
  name), `Vehicle_Type_Group` (a broad grouping of vehicle types by shared risk
  profile), and `Vehicle_Class` (an engine-size band, where engine capacity data is
  available).
- **Why it matters:** Phase 4's EDA highlighted that vehicle type strongly reflects
  occupant vulnerability; grouping raw vehicle types into a small number of
  risk-relevant categories (Motorcycle / Cycle / Car / Van / Heavy Vehicle) captures
  this pattern more robustly than the high-cardinality raw field.
- **Numeric vs. categorical missing-value handling:** `Vehicle_Type_Group` and
  `Vehicle_Class` map missing input to `"Unknown"` by construction, but `Vehicle_Age`
  is a **numeric copy** of `Age_of_Vehicle`, so any missing values in the source
  column carry over directly — this is the one feature in this section that needed an
  explicit `safety_net_impute()` call, now added above.


---
## 10. Binary Indicator Features

**Purpose:** Create simple, interpretable binary (0/1) indicator columns that flag
conditions repeatedly found relevant during Phase 4's EDA: weekend timing, night-time
driving, high-speed roads, and urban location.


In [22]:
def engineer_binary_indicators(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Create binary indicator columns: Is_Weekend, Is_Night, High_Speed_Road,
    and Urban_Area, from previously engineered or existing columns.

    Args:
        df (pd.DataFrame): The dataframe to add binary indicators to.

    Returns:
        pd.DataFrame: The dataframe with new binary indicator columns added.
    '''
    # Is_Weekend was already created (and safety-net imputed) in Section 3
    # if 'Date' was available; this branch only handles the case where it
    # wasn't created there.
    if "Is_Weekend" not in df.columns and "Day_of_Week" in df.columns:
        df["Is_Weekend"] = df["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)
        print("Created: 'Is_Weekend' (from 'Day_of_Week').")
    elif "Is_Weekend" in df.columns:
        print("'Is_Weekend' already created in Section 3 — not duplicated.")

    if "Time_of_Day" in df.columns:
        df["Is_Night"] = (df["Time_of_Day"] == "Night").astype(int)
        print("Created: 'Is_Night' (from 'Time_of_Day').")
    elif column_exists(df, "Light_Group"):
        df["Is_Night"] = (df["Light_Group"] == "Night").astype(int)
        print("Created: 'Is_Night' (from 'Light_Group', as a fallback).")

    if column_exists(df, "Speed_limit"):
        df["High_Speed_Road"] = (df["Speed_limit"] >= 60).astype(int)
        print("Created: 'High_Speed_Road' (Speed_limit >= 60 mph).")
        df = safety_net_impute(df, "High_Speed_Road", "source 'Speed_limit' was missing for these rows")

    if "Urban_Rural_Group" in df.columns:
        df["Urban_Area"] = (df["Urban_Rural_Group"] == "Urban").astype(int)
        print("Created: 'Urban_Area' (from 'Urban_Rural_Group').")
    elif column_exists(df, "Urban_or_Rural_Area"):
        df["Urban_Area"] = (df["Urban_or_Rural_Area"].astype(str).str.lower() == "urban").astype(int)
        print("Created: 'Urban_Area' (from 'Urban_or_Rural_Area', as a fallback).")

    return df


df = engineer_binary_indicators(df)


'Is_Weekend' already created in Section 3 — not duplicated.
Created: 'Is_Night' (from 'Time_of_Day').
Created: 'High_Speed_Road' (Speed_limit >= 60 mph).
Created: 'Urban_Area' (from 'Urban_Rural_Group').


In [23]:
binary_columns = [col for col in ["Is_Weekend", "Is_Night", "High_Speed_Road", "Urban_Area"] if col in df.columns]

for col in binary_columns:
    counts = df[col].value_counts().sort_index()
    percentage_positive = (df[col].mean() * 100) if df[col].notna().any() else 0
    print(f"{col}: {counts.to_dict()}  ->  {percentage_positive:.2f}% positive (1)")


Is_Weekend: {0: 2066400, 1: 649540}  ->  23.92% positive (1)
Is_Night: {0: 2399027, 1: 316913}  ->  11.67% positive (1)
High_Speed_Road: {0: 2073814, 1: 642126}  ->  23.64% positive (1)
Urban_Area: {0: 993995, 1: 1721945}  ->  63.40% positive (1)


**Interpretation**

- **What was created:** Four compact binary flags — `Is_Weekend`, `Is_Night`,
  `High_Speed_Road`, and `Urban_Area` — each summarizing a condition that Phase 4's EDA
  repeatedly surfaced as relevant to accident context.
- **Why it matters:** Binary indicators are maximally interpretable and computationally
  cheap, and often let simpler model families capture threshold-like effects (e.g.,
  "is this a high-speed road, yes/no") more directly than a multi-category feature would.
- `Is_Night` and `Urban_Area` are derived from already fully-populated categorical
  columns (`Time_of_Day`/`Light_Group`, `Urban_Rural_Group`), so they inherit full
  coverage automatically; `High_Speed_Road` is derived from the numeric `Speed_limit`
  and so receives its own explicit safety net, consistent with the numeric-vs-
  categorical distinction established in Section 9.


---
## 11. Feature Validation

**Purpose:** Programmatically identify every new feature created in this notebook, and
validate its data type, missing value count, and summary statistics — the primary
safeguard requested for this corrected version of the notebook.


In [24]:
new_features = [col for col in df.columns if col not in original_columns]

print(f"Total new features created: {len(new_features)}")
print()
for feature in new_features:
    print(f"  - {feature}")


Total new features created: 20

  - Year
  - Month
  - Day
  - Quarter
  - Is_Weekend
  - Hour
  - Minute
  - Time_of_Day
  - Speed_Category
  - Road_Category
  - Junction_Group
  - Weather_Group
  - Light_Group
  - Urban_Rural_Group
  - Vehicle_Age
  - Vehicle_Type_Group
  - Vehicle_Class
  - Is_Night
  - High_Speed_Road
  - Urban_Area


In [25]:
feature_validation = pd.DataFrame(
    {
        "Feature": new_features,
        "Data Type": [str(df[col].dtype) for col in new_features],
        "Missing Values": [int(df[col].isnull().sum()) for col in new_features],
        "Missing Percentage (%)": [
            round((df[col].isnull().sum() / len(df)) * 100, 2) for col in new_features
        ],
        "Unique Values": [df[col].nunique(dropna=True) for col in new_features],
    }
)
feature_validation


,Feature,Data Type,Missing Values,Missing Percentage (%),Unique Values
0,Year,float64,0,0.00,13
1,Month,object,0,0.00,12
2,Day,float64,0,0.00,12
3,Quarter,float64,0,0.00,4
4,Is_Weekend,int64,0,0.00,2
5,Hour,int64,0,0.00,24
6,Minute,int64,0,0.00,60
7,Time_of_Day,object,0,0.00,4
8,Speed_Category,object,0,0.00,3
9,Road_Category,object,0,0.00,4


In [26]:
numeric_new_features = [col for col in new_features if pd.api.types.is_numeric_dtype(df[col])]

if numeric_new_features:
    print("Summary statistics for new numerical features:")
    df[numeric_new_features].describe().T
else:
    print("No new numerical features to summarize.")


Summary statistics for new numerical features:


**Interpretation**

- **What it shows:** A complete, programmatically derived inventory of every feature
  added in this notebook, along with its data type, missing-value profile, and (for
  numerical features) descriptive statistics.
- Deriving `new_features` as a set difference from `original_columns` (rather than a
  hardcoded list) guarantees this validation table always reflects the actual notebook
  output, even if a particular source column was unavailable and a planned feature was
  skipped.


---
## Pre-Save Validation — Full Missing-Value Audit

**Purpose:** Perform one final, whole-dataset missing-value audit — across **every**
column, not just the newly engineered ones — immediately before saving, as an explicit
safety gate. This is the direct, requested check that would have caught the original
`Year`/`Day`/`Quarter` bug before it ever reached `featured_accident_data.csv`.


In [27]:
print(df.isna().sum().sort_values(ascending=False).head(20))


Date                                           1632422
1st_Road_Class                                       0
Accident_Severity                                    0
Day_of_Week                                          0
Did_Police_Officer_Attend_Scene_of_Accident          0
Junction_Control                                     0
Junction_Detail                                      0
Latitude                                             0
Light_Conditions                                     0
Local_Authority_(District)                           0
Longitude                                            0
Number_of_Vehicles                                   0
Pedestrian_Crossing-Human_Control                    0
Pedestrian_Crossing-Physical_Facilities              0
Road_Surface_Conditions                              0
Road_Type                                            0
Speed_limit                                          0
Time                                                 0
Urban_or_R

In [28]:
total_missing = int(df.isnull().sum().sum())
columns_with_missing = df.columns[df.isnull().any()].tolist()

if total_missing == 0:
    print("\nConfirmed: the dataset contains ZERO missing values across all columns.")
else:
    print(
        f"\n[WARNING] {total_missing} missing value(s) remain across "
        f"{len(columns_with_missing)} column(s): {columns_with_missing}"
    )
    print(
        "These are pre-existing, non-engineered columns carried over from earlier "
        "phases (not touched by this notebook's feature engineering logic) and are "
        "left as-is here, consistent with this notebook's scope; any handling "
        "decision for them belongs to Phase 3 (preprocessing), not this notebook."
    )

assert not df[new_features].isnull().any().any(), (
    "One or more ENGINEERED features still contain missing values — this must be "
    "zero before saving. Investigate the corresponding derivation/safety-net logic "
    "above before proceeding."
)
print("\nConfirmed: zero missing values across all ENGINEERED features specifically.")



[WARNING] 1632422 missing value(s) remain across 1 column(s): ['Date']
These are pre-existing, non-engineered columns carried over from earlier phases (not touched by this notebook's feature engineering logic) and are left as-is here, consistent with this notebook's scope; any handling decision for them belongs to Phase 3 (preprocessing), not this notebook.

Confirmed: zero missing values across all ENGINEERED features specifically.


**Interpretation**

- The first check above (`df.isna().sum().sort_values(ascending=False).head(20)`) is
  exactly the validation requested: a full, whole-dataset ranked missing-value audit,
  run immediately before saving.
- The second check narrows specifically to the columns this notebook is responsible
  for — the engineered features — and **asserts** zero missing values among them. This
  is deliberately stricter than the whole-dataset check: a pre-existing gap in an
  untouched, non-engineered column (e.g., a column already carrying some missingness
  from Phase 3) is not a regression introduced by this notebook, but any gap in a
  column **this notebook created** would be exactly the class of bug being fixed here,
  so it is treated as a hard failure via `assert`, not just a printed warning.


---
## 12. Save Dataset

**Purpose:** Persist the engineered dataset to
`Dataset/processed/featured_accident_data.csv`, creating the destination folder
automatically if it does not already exist.


In [29]:
PROCESSED_DATA_DIR = Path("..") / "Dataset" / "processed"
OUTPUT_FILE = PROCESSED_DATA_DIR / "featured_accident_data.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Engineered dataset saved successfully to: {OUTPUT_FILE.resolve()}")
    print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
except OSError as save_error:
    print(f"[ERROR] Failed to save engineered dataset: {save_error}")
    raise


Engineered dataset saved successfully to: C:\Users\Stalin\OneDrive\Desktop\draftrajproject\road-accident-severity-prediction\road-accident-severity-prediction\Dataset\processed\featured_accident_data.csv
Final shape: 2,715,940 rows x 58 columns


**Interpretation**

- The output directory is created programmatically with `Path.mkdir(parents=True,
  exist_ok=True)`, so the notebook runs successfully whether or not
  `Dataset/processed/` already exists (it does, from Phase 3, but this keeps the
  notebook self-contained).
- Saving with `index=False` avoids introducing a spurious unnamed index column into the
  engineered CSV, consistent with the convention established in Phase 3.
- Because Section 11 and the Pre-Save Validation gate above both ran (and passed)
  before this cell, this save is now guaranteed — not merely assumed — to be free of
  the `Year`/`Day`/`Quarter` missing-value bug reported previously.


---
## 13. Feature Engineering Summary

**Purpose:** Consolidate every engineered feature, its rationale, and its expected
impact on prediction into a single reference table for use in subsequent phases and
the final research report.


In [30]:
print(f"Original columns  : {len(original_columns)}")
print(f"New features added : {len(new_features)}")
print(f"Final columns      : {df.shape[1]}")


Original columns  : 38
New features added : 20
Final columns      : 58


| Feature | Reason for Creation | Expected Impact on Prediction |
|---|---|---|
| `Year` (cross-validated) | Captures the calendar year of the accident; now reconciled against `Date` rather than reused blindly. | Helps the model account for year-over-year drift in accident patterns. |
| `Month` | Captures seasonal variation identified in Phase 4 EDA (e.g., darker, wetter winter months). | May help the model capture weather/daylight-correlated seasonal risk. |
| `Day` | Day-of-month component of the accident date. | Minor standalone value; mainly supports other date-derived features. |
| `Quarter` | Coarser seasonal grouping than month, useful for reducing cardinality. | Provides a lower-cardinality alternative to `Month` for simpler models. |
| `Weekday` (if newly derived) | Named day of the week for the accident. | Supports weekday/weekend pattern recognition alongside `Is_Weekend`. |
| `Is_Weekend` | Binary flag distinguishing weekend from weekday accidents. | Captures leisure-travel and altered traffic-mix risk patterns found in EDA. |
| `Hour` | Clock hour of the accident. | Supports fine-grained time-of-day pattern recognition. |
| `Minute` | Clock minute of the accident. | Minor granularity; mainly supports `Hour`/`Time_of_Day` derivation. |
| `Time_of_Day` | Readable bucket (Morning/Afternoon/Evening/Night) summarizing `Hour`. | Reduces 24 raw hour values to 4 interpretable, EDA-informed risk windows. |
| `Speed_Category` | Groups discrete `Speed_limit` values into readable low/mid/high bands. | Expected strong predictor, reflecting the speed-severity link seen in EDA. |
| `Road_Category` | Groups `Road_Type` (+ road class) into Major Road / Minor Road / Roundabout / Other. | Captures typical traffic speed/volume context in a low-cardinality form. |
| `Junction_Group` | Flags whether the accident occurred at a junction. | Junctions concentrate conflicting vehicle paths, a known collision-type driver. |
| `Weather_Group` | Groups raw, sometimes-combined weather text into 6 clear categories. | Reduces sparsity in rare adverse-weather categories (fog, snow) for more reliable learning. |
| `Light_Group` | Distinguishes Day / Artificial Lighting / Night / Unknown from raw light text. | Preserves the lit-vs-unlit darkness distinction found relevant in EDA. |
| `Urban_Rural_Group` | Standardizes urban/rural labeling across text and legacy numeric codes. | Provides a robust source for the `Urban_Area` binary indicator. |
| `Vehicle_Age` | Renamed/retained from `Age_of_Vehicle` for clarity; safety-net imputed if missing. | Older vehicles may lack modern safety features, a plausible severity factor. |
| `Vehicle_Type_Group` | Groups raw vehicle types by shared occupant-vulnerability profile. | Expected to strongly separate high-vulnerability (motorcycle/cycle) records. |
| `Vehicle_Class` | Bins engine capacity into Small/Medium/Large bands. | Proxies vehicle size/mass, a plausible factor in collision energy and severity. |
| `Is_Night` | Binary flag for `Time_of_Day == 'Night'` (or `Light_Group == 'Night'` as fallback). | Directly encodes the unlit night-time risk window highlighted in EDA. |
| `High_Speed_Road` | Binary flag for `Speed_limit >= 60` mph; safety-net imputed if missing. | Directly encodes the high-speed collision-energy risk window highlighted in EDA. |
| `Urban_Area` | Binary flag for `Urban_Rural_Group == 'Urban'`. | Simple, direct complement to the well-established rural-severity relationship. |

> **Note:** The exact set of rows present in the table above corresponds to the
> features this notebook is designed to create; the actual notebook run will only
> include the subset whose source columns were available in your specific cleaned
> dataset (see Section 11's programmatically generated feature list for the ground
> truth of what was actually created in your run).

---

### Summary of the Bug Fix

| Aspect | Previous (buggy) behavior | Corrected behavior |
|---|---|---|
| Date parsing | Single `errors="coerce"` pass, no diagnostics, no fallback format | Two-pass parsing (dayfirst=True, then dayfirst=False fallback) with full diagnostic reporting of failures |
| `Year` handling | Reused blindly if already present, never checked for missing values | Always cross-validated against parsed `Date`; gaps filled from whichever source is valid |
| Residual missingness | Not checked; silently saved into the final CSV | Explicit `safety_net_impute()` on every date/time feature, plus an `assert` immediately after derivation |
| Pre-save check | None | Whole-dataset `isna().sum().sort_values(ascending=False).head(20)` audit, plus a hard `assert` that zero missing values remain among engineered features specifically |

### Next Steps

The next notebook (`05_Feature_Selection_and_Data_Preparation.ipynb`) can now be
re-run against this corrected `featured_accident_data.csv` without encountering the
`NaN`-in-`X_train` failure that motivated this fix.
